In [31]:
endpoint = "https://circlektesting.openai.azure.com/"
model_name = "text-embedding-3-small"
deployment = "invoice-embeddings"

api_version = "2024-02-01"

In [32]:
from azure.identity import DeviceCodeCredential, get_bearer_token_provider
from openai import AzureOpenAI

credential = DeviceCodeCredential(
    tenant_id="3e41b164-59e6-4ce9-8c15-767e2c81431c",
)
token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default",
)

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    azure_ad_token_provider=token_provider,
)

In [33]:
item_types = [
    "ADVERTISING MATERIALS (SIGNAGE, STICKERS, ETC.)",
    "ADVERTISING SERVICES LABOR ONLY",
    "ADVERTISING DIGITAL ADVERTISING SERVICES (Online Ads)",
    "COMPUTER HARDWARE",
    "PREWRITTEN COMPUTER SOFTWARE CANNED or LICENSE",
    "CUSTOM COMPUTER SOFTWARE",
    "COMPUTER SOFTWARE MAINTENANCE CONTRACTS OPTIONAL",
    "COMPUTER SOFTWARE MAINTENANCE CONTRACTS REQUIRED",
    "SOFTWARE AS A SERVICE (SaaS)",
    "INFRASTRUCTURE AS A SERVICE (IaaS)",
    "PLATFORM AS A SERVICE (PaaS)",
    "ARCHITECT/ ENGINEER SERVICES CONCEPT SKETCH, REMODEL DRAWINGS, ETC.",
    "REAL ESTATE MATERIALS PERMANENTLY AFFIXED OR INCORPORATED INTO BUILDING (CANOPY, DOOR, HVAC, PLUMBING, ROOF, WINDOW, ETC.",
    "INSPECTION SERVICES",
    "TANGIBLE PERSONAL PROPERTY ITEMS REMAIN TANGIBLE RACK, REFRIGERATOR, STORE EQUIPMENT ETC.",
    "TANGIBLE PERSONAL PROPERTY LABOR: INSTALLATION",
    "TANGIBLE PERSONAL PROPERTY LABOR: REPAIRS/ RESTORE/ SERVICING",
    "MATERIAL / PARTS / TOOLS / EQUIPMENT FOR REPAIRING",
    "EQUIPMENT - FOOD PREPARATION REFRIGERATOR MUST FOR FOOD & KITCHEN AND NOT FOR STORAGE OR DISPLAY, GRILL, ICE MAKER, MICROWAVES, OVENS ETC. Manufacturing States - OH-IN-MN-TX",
    "EQUIPMENT-FOOD SERVING SPOONS, TONGS, ETC.",
    "EQUIPMENT-FOOD STORAGE BINS, REFRIGERATOR, ETC.",
    "LANDSCAPING - LABOR",
    "PLUMBING INSTALLATION SERVICES ASSUME PLUMBER PAID SALES TAX ON FIXTURES WHEN PURCHASED",
    "PROFESSIONAL SERVICES ACCOUNTING & FINANCIAL, ADVERTISING, ARCHITECTS, CONSULTING, ENGINEERING, IT, LEGAL, MARKETING, ETC.",
    "STORAGE SERVICES",
    "FURNITURE & FIXTURES KITCHEN, DINING & OFFICE",
    "INTERNET ACCESS",
    "INVENTORY WITHDRAWAL CUPS, PAPER TOWELS, UTENSILS, ETC.",
    "LEASED EQUIPMENT LEASING FROM 3RD PARTY",
    "LEASE-REAL PROPERTY LEASING FROM 3RD PARTY",
    "FILMS & FOILS FOIL PANS/LIDS, PLASTIC WRAP",
    "PAPER PRODUCTS NAPKINS, STRAWS, STIRRERS, ETC.",
    "PLASTIC - DISPOSABLE UTENSILS, BAGS, ETC.",
    "PLASTIC REUSABLE",
    "SMALL WARES-KITCHEN PANS, METAL TRAYS, ETC.",
    "SMALL WARES-TABLETOP SALT/PEPPER SHAKERS, MUSTARD, ETC.",
    "SUPPLIES-OFFICE PAPER, PENS, ETC.",
    "SUPPLIES-STORE REGISTER TAPE, PENS, ETC.",
    "FREIGHT",
    "PACKING / HANDLING"
]

In [36]:
line_item = "GILM14330A001 VERIFONE CARD READER - UX300 Miscellaneous"

In [37]:
import math

from openai import PermissionDeniedError

try:
    response = client.embeddings.create(
        input=[line_item, *item_types],
        model=deployment,
    )
except PermissionDeniedError:
    raise RuntimeError(
        "The signed-in account needs the Cognitive Services OpenAI User role on circlektesting."
    ) from None

embeddings = [item.embedding for item in sorted(response.data, key=lambda item: item.index)]
line_item_embedding = embeddings[0]

def cosine_similarity(left, right):
    dot_product = sum(a * b for a, b in zip(left, right))
    left_norm = math.sqrt(sum(value * value for value in left))
    right_norm = math.sqrt(sum(value * value for value in right))
    return dot_product / (left_norm * right_norm)

ranked_item_types = sorted(
    (
        (cosine_similarity(line_item_embedding, embedding), item_type)
        for item_type, embedding in zip(item_types, embeddings[1:])
    ),
    reverse=True,
)

for rank, (score, item_type) in enumerate(ranked_item_types[:3], start=1):
    print(f"{rank}. {item_type} (cosine similarity: {score:.4f})")

1. COMPUTER HARDWARE (cosine similarity: 0.3768)
2. CUSTOM COMPUTER SOFTWARE (cosine similarity: 0.3163)
3. SUPPLIES-STORE REGISTER TAPE, PENS, ETC. (cosine similarity: 0.2911)
